In [ ]:
import gzip, pickle, os, numpy as np, sys
import hpc_optimization_job, optimize_sheet_cli, visualization, mesh
sys.path.append('optimization_jobs')

In [ ]:
import lilium_job_config as config

In [ ]:
jobs = hpc_optimization_job.ParameterSweepJobs(config)

In [ ]:
job_index = 0

In [ ]:
paramsWithChoices = jobs.paramsWithChoices()#[n for n, v in config.parameterChoices.items() if len(v) > 1]
choicelessParams = jobs.choicelessParams()
analysisDir = 'SiggraphExamples/results/analysis'
frameChoice = 'cbThreshold'

In [ ]:
import inspect, json
def generateFlipper(title, views, frames, statistics = [], framesLabel = None):
    framesLabelString = '' if framesLabel is None else f"framesLabel = '{framesLabel}';\n"
    framesLabelString = '' if framesLabel is None else f"framesLabel = '{framesLabel}';\n"
    return inspect.cleandoc(f"""
    title = '{title}';
    {framesLabelString}statistics = {json.dumps(statistics)};
    views = {json.dumps(views)};
    frames = {json.dumps(frames)};
    """)

In [ ]:
import collections
from matplotlib import pyplot as plt
os.makedirs(analysisDir, exist_ok=True)

if frameChoice is None: frameChoice = paramsWithChoices[-1]
ignoreParams = [frameChoice] + choicelessParams

flipperContents = collections.defaultdict(list)
flipperTitle = {}
for ji in range(jobs.numJobs()):
    if optimize_sheet_cli.check(jobs, ji) != 'SUCCESS': continue
    params = jobs.parametersForJob(ji)
    flipperName = jobs.jobName(ji, param_sep=':', value_sep='_', ignoreParams=ignoreParams, useShortName = False)
    flipperTitle[flipperName] = jobs.jobName(ji, param_sep=', ', value_sep=' ', ignoreParams=ignoreParams, useShortName = False)
    flipper_data_dir = jobs.jobName(ji, ignoreParams=choicelessParams)
    images = [f'{flipper_data_dir}/{i}' for i in ['convergence_plots.png', 'opt_design.jpg', 'opt_deploy.jpg']]# 'equilibrium.jpg', 'equilibrium_with_target.jpg']]
    frame = {'image': images, 'name': params[frameChoice]}
    lf_report = pickle.load(gzip.open(jobs.directoryForJob(ji) + '/lowfit_opt_report.pkl.gz', 'r'))
    frame.update({'Final Energy ' + k: str(v) for k, v in lf_report.energies[-1].items()})
    frame.update({'Final Grad Norm': str(lf_report.gradNorms[-1])})
    flipperContents[flipperName].append(frame)

In [ ]:
views = ['Convergence Plot', 'Optimized Design', 'Equilibrium']#, 'Equilibrium vs Target']
stats = ['Final Energy ' + t for t in ['Full', 'Fitting', 'CollapseBarrier', 'Smoothing']]
flipperDirectory = []
for flipperName, flipperData in flipperContents.items():
    flipper_data_dir = jobs.jobName(ji, ignoreParams=ignoreParams)
    absolute_flipper_data_dir = f'{analysisDir}/{flipper_data_dir}'
    flipperPath = f'{analysisDir}/{flipperName}.js'
    print(generateFlipper(flipperTitle[flipperName], views, flipperData, framesLabel=frameChoice, statistics=stats), file=open(flipperPath, 'w'))
    flipperDirectory.append([flipperName, flipperName + '.js'])
print(f'flippers = {json.dumps(flipperDirectory)};', file=open(f'{analysisDir}/directory.js', 'w'))